In [1]:
# ═══════════════════════════════════════════════════════════════
# VOLUME DISCOVERY SCANNER — S&P 500 + NASDAQ 100 + EXTRAS
# ═══════════════════════════════════════════════════════════════
#
# Purpose:   Surface stocks printing abnormal volume RIGHT NOW
#            across ~600 names — finds what you don't already
#            have on a watchlist
#
# Examples this is built for:
#   LCID  → dead all morning, news at 2pm, volume explodes
#   SMCI  → accounting fraud, massive volume at the open
#
# ─────────────────────────────────────────────────────────────
# HOW IT SCORES
# ─────────────────────────────────────────────────────────────
# Volume discovery:  -3x volume vs the ticker's last five sessions
#                   -all-session (regular hours only) MEDIAN
#                   "busy, period"
#                       Why median and not mean:
#                       One outlier day in the lookback barely moves
#                       a median. It wrecks a mean.
#
#   Baseline: median 2-MIN bar volume for that ticker
#             across the last 5 sessions, 9:46AM-4PM
#             Computed once at startup
#
#   Score:    current bar volume ÷ baseline
#
#   Why prior-day baseline and not intraday:
#     A stock that gapped on news is heavy ALL DAY.
#     An intraday baseline would treat that as its
#     new normal and the stock would vanish from
#     the board — exactly backwards for discovery.
#     Prior-day baseline means SMCI's fraud-day open
#     scores enormous against its quiet prior week.
#
#
# ─────────────────────────────────────────────────────────────
# LEADERBOARD, NOT ALERTS
# ─────────────────────────────────────────────────────────────
#
#   600 tickers would generate dozens of alerts a day.
#   A firehose you learn to ignore is worse than nothing.
#
#   Instead: top 10 most unusual RIGHT NOW,
#   re-ranked every 2 minutes. You glance at it.
#   New entries flash. No beep, no popup.
#
#   Click any row → ticker copied to clipboard
#
# ─────────────────────────────────────────────────────────────
# UNIVERSE
# ─────────────────────────────────────────────────────────────
#
#   SP500.txt       ← refresh quarterly
#   NDX100.txt      ← refresh quarterly
#   VOL_extras.txt  ← IPOs, leveraged ETFs, anything you want
#
#   Deduped. Price floor $4 — below that is usually
#   untradeable noise even at 50x volume.
#
# ─────────────────────────────────────────────────────────────
# TIMING
# ─────────────────────────────────────────────────────────────
#
#   9:00AM  run startup → builds baselines (~60s)
#   9:46AM  scoring begins (opening bars excluded)
#   10:06AM leaderboard fully populated
#   4:00PM  session ends
#
#   Scans fire 5s after each 2-MIN bar close
#
# ─────────────────────────────────────────────────────────────
# LOG
# ─────────────────────────────────────────────────────────────
#
#   VOL_discovery_log.csv
#   Logs the FIRST time each ticker hits the board each day.
#   Review at 4PM — did anything here turn out tradeable?
#
# ═══════════════════════════════════════════════════════════════

%pip install alpaca-py
%pip install pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ─── Cell 2: Imports ─────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import time as time_module
import threading
import datetime as dt
from datetime import datetime
from dotenv import load_dotenv

import tkinter as tk

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit

print("✅ Libraries imported")

✅ Libraries imported


In [3]:
# ─── Cell 3: Alpaca Connection ────────────────────────────────
dotenv_path = (
    r"c:\Users\kwgit\Desktop\Playbook"
    r"\Claude_2025_2026_Playbooks"
    r"\AlgoTrading\AlgoTrading\.env"
)
load_dotenv(dotenv_path, override=True)

data_client = StockHistoricalDataClient(
    api_key    = os.getenv("ALPACA_API_KEY"),
    secret_key = os.getenv("ALPACA_SECRET_KEY")
)

print("✅ Alpaca data client connected")

✅ Alpaca data client connected


In [4]:
# ─── Cell 4: Configuration ────────────────────────────────────

BASE_PATH = (
    r"c:\Users\kwgit\Desktop\Playbook"
    r"\Claude_2025_2026_Playbooks"
    r"\AlgoTrading\AlgoTrading"
)

SP500_FILE   = BASE_PATH + r"\SP500.txt"
NDX100_FILE  = BASE_PATH + r"\NDX100.txt"
EXTRAS_FILE  = BASE_PATH + r"\VOL_extras.txt"
LOG_FILE     = BASE_PATH + r"\VOL_discovery_log.csv"

# ── Timing ────────────────────────────────────────────────────
TRACK_START   = dt.time(9, 46)
SESSION_END   = dt.time(16, 0)
BAR_BUFFER    = 5

# ── Baseline ──────────────────────────────────────────────────
BASELINE_DAYS = 5
MIN_BASE_BARS = 100
MIN_SLOT_OBS  = 3
ACCEL_WINDOW  = 15

# ── Filters ───────────────────────────────────────────────────
PRICE_FLOOR   = 4.00
MIN_SCORE     = 3.0      # normal eligibility floor
MIN_SCORE_HOD = 2.0      # lower floor when the bar is
                         # the ticker's biggest of the day —
                         # "biggest bar today" is meaningful
                         # even at a modest historical score
MIN_VOLUME    = 10_000

# ── Leaderboard ───────────────────────────────────────────────
BOARD_ROWS    = 10
BATCH_SIZE    = 200

# ── Voice ─────────────────────────────────────────────────────
VOICE_ON        = True
VOICE_THRESHOLD = 15.0   # speak at 15x+ only
                         # every board entry would train
                         # you to tune it out within a week
VOICE_RATE      = 0      # -10 slow … 10 fast (SAPI)

print(f"✅ Configuration loaded")
print(f"   Baseline:     {BASELINE_DAYS} sessions, "
      f"per-slot median")
print(f"   Track from:   {TRACK_START}")
print(f"   Scan timing:  bar close + {BAR_BUFFER}s")
print(f"   Price floor:  ${PRICE_FLOOR:.2f}")
print(f"   Min score:    {MIN_SCORE}x "
      f"({MIN_SCORE_HOD}x if HOD)")
print(f"   Voice:        "
      f"{'on at ' + str(VOICE_THRESHOLD) + 'x' if VOICE_ON else 'off'}")

✅ Configuration loaded
   Baseline:     5 sessions, per-slot median
   Track from:   09:46:00
   Scan timing:  bar close + 5s
   Price floor:  $4.00
   Min score:    3.0x (2.0x if HOD)
   Voice:        on at 15.0x


In [5]:
# ─── Cell 5: Universe Loader ──────────────────────────────────

def read_ticker_file(path, label):
    """Read one ticker file, skip # comments and blanks"""
    if not os.path.exists(path):
        print(f"   ⚠️  {label}: not found at {path}")
        return []

    with open(path, 'r') as f:
        tickers = [
            line.strip().upper()
            for line in f
            if line.strip() and
            not line.strip().startswith('#')
        ]
    print(f"   {label:<10} {len(tickers):>4} tickers")
    return tickers


def load_universe():
    """
    Combine S&P 500 + Nasdaq 100 + extras, deduped

    Extras file is created empty if missing —
    that's where IPOs and leveraged ETFs go
    """
    print("Loading universe...")

    sp500  = read_ticker_file(SP500_FILE,  "SP500")
    ndx    = read_ticker_file(NDX100_FILE, "NDX100")

    if not os.path.exists(EXTRAS_FILE):
        with open(EXTRAS_FILE, 'w') as f:
            f.write("# Extra tickers to scan\n")
            f.write("# IPOs, leveraged ETFs, anything\n")
            f.write("# not in the S&P or Nasdaq lists\n")
            f.write("# One per line\n")
        print(f"   ✅ Created VOL_extras.txt")
        extras = []
    else:
        extras = read_ticker_file(EXTRAS_FILE, "Extras")

    universe = sorted(set(sp500 + ndx + extras))

    # Alpaca chokes on dot notation for share classes
    universe = [t for t in universe if '.' not in t]

    print(f"\n   ✅ Universe: {len(universe)} unique tickers")
    return universe


# ── Log setup ─────────────────────────────────────────────────
log_headers = [
    'timestamp', 'ticker', 'bar_time',
    'score', 'accel', 'volume', 'baseline',
    'close', 'pct_change', 'direction',
    'peak_score', 'notes'
]

if not os.path.exists(LOG_FILE):
    pd.DataFrame(columns=log_headers).to_csv(
        LOG_FILE, index=False)
    print(f"\n✅ Created VOL_discovery_log.csv")
else:
    existing = pd.read_csv(LOG_FILE)
    print(f"\n✅ Log found — {len(existing)} entries")

universe = load_universe()


✅ Log found — 130 entries
Loading universe...
   ⚠️  SP500: not found at c:\Users\kwgit\Desktop\Playbook\Claude_2025_2026_Playbooks\AlgoTrading\AlgoTrading\SP500.txt
   NDX100      102 tickers
   Extras       23 tickers

   ✅ Universe: 124 unique tickers


In [6]:
# ─── Cell 6: Baseline Builder ─────────────────────────────────

# {ticker: {'slots': {…}, 'flat': float,
#           'prev_close': float, 'bars': int}}
baselines = {}

# Universe-wide volume shape: {'HH:MM': factor}
# Used as fallback when a ticker's slot is too thin
slot_shape = {}


def fetch_batch(tickers, start, timeframe_mins=2):
    """
    Fetch bars for a batch of tickers
    Returns a single DataFrame with a symbol column
    """
    try:
        req = StockBarsRequest(
            symbol_or_symbols = tickers,
            timeframe          = TimeFrame(
                timeframe_mins, TimeFrameUnit.Minute),
            start = start,
        )
        bars = data_client.get_stock_bars(req)
        df   = bars.df.reset_index()

        if len(df) == 0:
            return None

        df['timestamp'] = pd.to_datetime(df['timestamp'])
        if df['timestamp'].dt.tz is None:
            df['timestamp'] = df['timestamp'] \
                .dt.tz_localize('UTC')
        df['timestamp'] = df['timestamp'] \
            .dt.tz_convert('America/New_York')

        return df

    except Exception as e:
        print(f"      ❌ batch error: {str(e)[:60]}")
        return None


def build_baselines(tickers):
    """
    Build per-ticker, per-time-slot volume baselines

    WHY SLOTS AND NOT ONE FLAT MEDIAN:
      Volume follows a U-curve — heavy at the open,
      thin midday, heavy into the close. A flat median
      mixes all three, so an ordinary 9:46 bar scores
      4-5x automatically and the board fills with
      structural noise every morning.

      Comparing a 9:46 bar only against prior 9:46 bars
      removes that. An ordinary open scores ~1x.
      Real news at the open still stands out because
      everything around it drops to 1x.

    FALLBACK FOR THIN SLOTS:
      5 sessions = at most 5 observations per slot.
      Thin tickers skip slots entirely (Alpaca only
      returns a bar if something traded), so a slot
      may have 0-2 observations and an untrustworthy
      median.

      When that happens we use the ticker's flat median
      scaled by the universe-wide shape factor for that
      slot — borrowing the curve from liquid names.

    Also captures previous close for % change display.
    """
    global baselines, slot_shape
    baselines  = {}
    slot_shape = {}

    start = datetime.now() - dt.timedelta(
        days=BASELINE_DAYS + 4)   # pad for weekends
    today = datetime.now().date()

    n_batches = (len(tickers) + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"\nBuilding baselines — "
          f"{len(tickers)} tickers, {n_batches} batches")
    print(f"   Window: {TRACK_START} - {SESSION_END}, "
          f"last {BASELINE_DAYS} sessions")
    print(f"   Mode:   per-slot medians "
          f"(time-of-day bucketed)")

    t0 = time_module.time()
    all_frames = []

    for i in range(0, len(tickers), BATCH_SIZE):
        batch = tickers[i:i + BATCH_SIZE]
        bnum  = i // BATCH_SIZE + 1
        print(f"   Batch {bnum}/{n_batches} "
              f"({len(batch)} tickers)...", end=' ')

        df = fetch_batch(batch, start)
        if df is not None:
            all_frames.append(df)
            print(f"{len(df):,} bars")
        else:
            print("no data")

    if not all_frames:
        print("❌ No baseline data returned")
        return

    df_all = pd.concat(all_frames, ignore_index=True)

    # ── Filter to regular session, prior days only ────────────
    df_all['date'] = df_all['timestamp'].dt.date
    df_all['time'] = df_all['timestamp'].dt.time
    df_all['slot'] = df_all['timestamp'].dt.strftime('%H:%M')

    mask = (
        (df_all['time'] >= TRACK_START) &
        (df_all['time'] <  SESSION_END) &
        (df_all['date'] <  today)
    )
    df_hist = df_all[mask]

    if len(df_hist) == 0:
        print("❌ No historical bars in session window")
        return

    # ── Universe-wide shape curve ─────────────────────────────
    # Ratio of each slot's median to the overall median
    # Captures the U-curve so thin tickers can borrow it
    global_median = df_hist['volume'].median()
    slot_medians  = df_hist.groupby('slot')['volume'].median()

    if global_median > 0:
        slot_shape = (slot_medians / global_median).to_dict()

    # ── Per-ticker flat median and bar count ──────────────────
    flat_meds = df_hist.groupby('symbol')['volume'].median()
    bar_counts= df_hist.groupby('symbol')['volume'].count()

    # ── Per-ticker, per-slot medians and counts ───────────────
    slot_grp  = df_hist.groupby(['symbol', 'slot'])['volume']
    slot_med  = slot_grp.median()
    slot_cnt  = slot_grp.count()

    # ── Previous close per ticker ─────────────────────────────
    df_sorted  = df_hist.sort_values('timestamp')
    last_bars  = df_sorted.groupby('symbol').last()
    prev_close = last_bars['close']

    skipped_thin  = 0
    skipped_price = 0
    total_slots   = 0
    thin_slots    = 0

    for ticker in flat_meds.index:
        flat  = flat_meds[ticker]
        n     = bar_counts[ticker]
        close = prev_close.get(ticker, 0)

        if n < MIN_BASE_BARS:
            skipped_thin += 1
            continue
        if close < PRICE_FLOOR:
            skipped_price += 1
            continue
        if flat <= 0:
            continue

        # Build the slot dict for this ticker
        try:
            t_med = slot_med.loc[ticker]
            t_cnt = slot_cnt.loc[ticker]
        except KeyError:
            continue

        slots = {}
        for slot in t_med.index:
            total_slots += 1
            if t_cnt[slot] >= MIN_SLOT_OBS:
                slots[slot] = float(t_med[slot])
            else:
                thin_slots += 1
                # too few observations — skip it,
                # lookup falls back to shaped flat median

        baselines[ticker] = {
            'slots':      slots,
            'flat':       float(flat),
            'prev_close': float(close),
            'bars':       int(n)
        }

    elapsed = time_module.time() - t0

    print(f"\n   ✅ Baselines built in {elapsed:.0f}s")
    print(f"      Usable tickers:  {len(baselines)}")
    print(f"      Skipped (thin):  {skipped_thin}")
    print(f"      Skipped (<${PRICE_FLOOR:.0f}): "
          f"{skipped_price}")

    if total_slots > 0:
        pct = thin_slots / total_slots * 100
        print(f"      Slots w/ <{MIN_SLOT_OBS} obs: "
              f"{thin_slots:,} of {total_slots:,} "
              f"({pct:.0f}%) → shape fallback")

    if slot_shape:
        open_f  = slot_shape.get('09:46', 1)
        mid_f   = slot_shape.get('12:30', 1)
        close_f = slot_shape.get('15:56', 1)
        print(f"\n      Volume shape (× all-day median):")
        print(f"        09:46  {open_f:.2f}x")
        print(f"        12:30  {mid_f:.2f}x")
        print(f"        15:56  {close_f:.2f}x")


print("✅ Baseline builder loaded — time-of-day bucketed")
print("   build_baselines(universe) → run once at 9:00AM")

✅ Baseline builder loaded — time-of-day bucketed
   build_baselines(universe) → run once at 9:00AM


In [7]:
# ─── Cell 7: Live Scoring ─────────────────────────────────────

board         = []
prev_board    = set()
logged_today  = set()
printed_today = set()
spoken_today  = set()

# Highest 2-MIN volume seen today, per ticker
session_max   = {}

# Full day's history behind the log tab
# {ticker: {first_seen, last_seen, peak_score,
#           peak_accel, hod_hit, close, pct_change}}
day_log       = {}

# Last state written to CSV — skips redundant writes
last_peak_snapshot = {}


def reset_day():
    global board, prev_board, logged_today
    global printed_today, spoken_today
    global session_max, day_log, last_peak_snapshot
    board              = []
    prev_board         = set()
    logged_today       = set()
    printed_today      = set()
    spoken_today       = set()
    session_max        = {}
    day_log            = {}
    last_peak_snapshot = {}


def seconds_to_next_bar(buffer=BAR_BUFFER):
    """
    Seconds until the next 2-MIN boundary + buffer
    A bar timestamped 9:46 closes at 9:48
    """
    now    = datetime.now()
    mins   = now.minute
    nxt    = mins + (2 - mins % 2)
    target = now.replace(second=0, microsecond=0)
    target = target + dt.timedelta(
        minutes=(nxt - mins), seconds=buffer)
    delta  = (target - now).total_seconds()
    return delta if delta > 0 else delta + 120


def get_slot_baseline(ticker, slot):
    """
    Baseline for this ticker at this time slot

    1. Ticker's median for that exact slot
       (needs MIN_SLOT_OBS observations)
    2. Ticker's flat median × universe shape factor
    3. Ticker's flat median
    """
    base = baselines.get(ticker)
    if base is None:
        return None

    slot_val = base['slots'].get(slot)
    if slot_val and slot_val > 0:
        return slot_val

    factor = slot_shape.get(slot, 1.0)
    return base['flat'] * factor


def fetch_today_bars(tickers):
    """Today's bars for the full universe, batched"""
    start      = datetime.now() - dt.timedelta(days=1)
    all_frames = []

    for i in range(0, len(tickers), BATCH_SIZE):
        batch = tickers[i:i + BATCH_SIZE]
        df    = fetch_batch(batch, start)
        if df is not None:
            all_frames.append(df)

    if not all_frames:
        return None

    return pd.concat(all_frames, ignore_index=True)


def score_universe():
    """
    Score every ticker's latest CLOSED bar

    Three signals:

      score → bar ÷ same-slot median, 5 sessions
              "unusual for this time of day"

      accel → bar ÷ median of last 15 bars today
              "just changed, or elevated all day"

      hod   → biggest 2-MIN bar this ticker has
              printed today. Binary, not rankable

    HOD bars clear a lower eligibility floor —
    a ticker's biggest bar of the session is worth
    seeing even at a modest historical score
    """
    global session_max

    tickers = list(baselines.keys())
    if not tickers:
        return []

    df = fetch_today_bars(tickers)
    if df is None:
        return []

    today = datetime.now().date()
    now   = pd.Timestamp.now(tz='America/New_York')

    df['date'] = df['timestamp'].dt.date
    df['time'] = df['timestamp'].dt.time
    df['slot'] = df['timestamp'].dt.strftime('%H:%M')

    mask = (
        (df['date'] == today) &
        (df['time'] >= TRACK_START) &
        (df['time'] <  SESSION_END)
    )
    df = df[mask]

    if len(df) == 0:
        return []

    # Drop in-progress bars — a bar timestamped 9:46
    # isn't complete until 9:48
    cutoff = now - pd.Timedelta(minutes=2)
    df     = df[df['timestamp'] <= cutoff]

    if len(df) == 0:
        return []

    df = df.sort_values('timestamp')

    # Rolling median for acceleration
    recent = df.groupby('symbol').tail(ACCEL_WINDOW + 1)
    accel_base = (recent.groupby('symbol')
                  .apply(lambda g: g['volume']
                         .iloc[:-1].median()
                         if len(g) > 1 else np.nan))

    # Max volume today per ticker, excluding latest bar
    prior_bars = df.groupby('symbol').apply(
        lambda g: g['volume'].iloc[:-1].max()
        if len(g) > 1 else 0)

    latest  = df.groupby('symbol').last()
    results = []

    for ticker, row in latest.iterrows():
        vol  = row['volume']
        slot = row['slot']

        if vol < MIN_VOLUME:
            continue
        if row['close'] < PRICE_FLOOR:
            continue

        base = get_slot_baseline(ticker, slot)
        if base is None or base <= 0:
            continue

        score = vol / base

        # ── HOD ───────────────────────────────────────
        pm     = prior_bars.get(ticker, 0)
        is_hod = bool(pm > 0 and vol > pm)
        session_max[ticker] = max(
            session_max.get(ticker, 0), vol)

        floor = MIN_SCORE_HOD if is_hod else MIN_SCORE
        if score < floor:
            continue

        ab    = accel_base.get(ticker, np.nan)
        accel = (vol / ab) if (pd.notna(ab) and ab > 0) \
                else 0

        prev = baselines[ticker]['prev_close']
        pct  = ((row['close'] - prev) / prev * 100) \
               if prev > 0 else 0

        results.append({
            'ticker':     ticker,
            'score':      round(score, 1),
            'accel':      round(accel, 1),
            'hod':        is_hod,
            'volume':     int(vol),
            'baseline':   int(base),
            'close':      round(row['close'], 2),
            'pct_change': round(pct, 1),
            'bar_time':   row['timestamp'].strftime('%H:%M'),
            'direction':  'up' if row['close'] >= row['open']
                          else 'down',
        })

    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:BOARD_ROWS]


def update_day_log(current):
    """
    Maintain full-day history behind the log tab

    One row per ticker, updated as it reappears.
    Peak score climbs, HOD sticks once hit.

    A ticker that hit 8x then climbed to 45x reads
    very differently from one that hit 8x and faded —
    that difference is what the log is for
    """
    global day_log

    for e in current:
        t = e['ticker']

        if t not in day_log:
            day_log[t] = {
                'ticker':     t,
                'first_seen': e['bar_time'],
                'last_seen':  e['bar_time'],
                'peak_score': e['score'],
                'peak_accel': e['accel'],
                'hod_hit':    e['hod'],
                'close':      e['close'],
                'pct_change': e['pct_change'],
                'volume':     e['volume'],
            }
        else:
            d = day_log[t]
            d['last_seen']  = e['bar_time']
            d['peak_score'] = max(d['peak_score'],
                                  e['score'])
            d['peak_accel'] = max(d['peak_accel'],
                                  e['accel'])
            d['hod_hit']    = d['hod_hit'] or e['hod']
            d['close']      = e['close']
            d['pct_change'] = e['pct_change']
            d['volume']     = e['volume']


def log_new_entries(current):
    """First appearance of each ticker → CSV"""
    global logged_today

    for entry in current:
        t = entry['ticker']
        if t in logged_today:
            continue

        logged_today.add(t)

        row = {
            'timestamp':  datetime.now().strftime(
                          '%Y-%m-%d %H:%M:%S'),
            'ticker':     t,
            'bar_time':   entry['bar_time'],
            'score':      entry['score'],
            'accel':      entry['accel'],
            'hod':        entry['hod'],
            'volume':     entry['volume'],
            'baseline':   entry['baseline'],
            'close':      entry['close'],
            'pct_change': entry['pct_change'],
            'direction':  entry['direction'],
            'peak_score': entry['score'],
            'notes':      ''
        }

        df_new = pd.DataFrame([row])
        try:
            if os.path.exists(LOG_FILE):
                df_old = pd.read_csv(LOG_FILE)
                df_out = (df_new if len(df_old) == 0
                          else pd.concat([df_old, df_new],
                                         ignore_index=True))
            else:
                df_out = df_new
            df_out.to_csv(LOG_FILE, index=False)
        except PermissionError:
            # CSV open in Excel — drop it from the
            # logged set so it retries next cycle
            logged_today.discard(t)
        except Exception as e:
            print(f"   ⚠️  log write: {str(e)[:40]}")


def flush_peaks():
    """
    Push current peaks to the CSV every cycle

    Skips the write entirely when nothing changed —
    no point rewriting the whole file every 2 minutes
    for identical data.

    PermissionError means the CSV is open in Excel.
    That cycle is skipped, next one picks it up.
    """
    global last_peak_snapshot

    if not os.path.exists(LOG_FILE) or not day_log:
        return

    snapshot = {t: (d['peak_score'], d['peak_accel'],
                    d['hod_hit'])
                for t, d in day_log.items()}

    if snapshot == last_peak_snapshot:
        return

    try:
        df = pd.read_csv(LOG_FILE)
        if len(df) == 0:
            return

        today = datetime.now().strftime('%Y-%m-%d')
        mask  = df['timestamp'].str.startswith(today)

        for t, d in day_log.items():
            sel = mask & (df['ticker'] == t)
            df.loc[sel, 'peak_score'] = d['peak_score']
            df.loc[sel, 'hod']        = d['hod_hit']
            df.loc[sel, 'accel']      = d['peak_accel']
            df.loc[sel, 'close']      = d['close']
            df.loc[sel, 'pct_change'] = d['pct_change']

        df.to_csv(LOG_FILE, index=False)
        last_peak_snapshot = snapshot

    except PermissionError:
        pass
    except Exception as e:
        print(f"   ⚠️  peak flush: {str(e)[:40]}")


print("✅ Scoring loaded — bucketed + accel + HOD")
print(f"   score ≥ {MIN_SCORE}x  ({MIN_SCORE_HOD}x if HOD)")
print(f"   Peaks flushed to CSV every cycle")

✅ Scoring loaded — bucketed + accel + HOD
   score ≥ 3.0x  (2.0x if HOD)
   Peaks flushed to CSV every cycle


In [ ]:
# ─── Cell 8: Tabbed Dashboard + Voice ────────────────────────
from tkinter import ttk
import queue

# ── Voice engine ──────────────────────────────────────────────
# Own thread — a blocking speak() inside the scan
# loop would stall the next scan

voice_queue = queue.Queue()


def voice_worker():
    """
    Background speech thread
    SAPI first (Windows native), pyttsx3 fallback
    Degrades silently if neither is available
    """
    speaker = None
    engine  = None

    try:
        import win32com.client
        speaker = win32com.client.Dispatch("SAPI.SpVoice")
        speaker.Rate = VOICE_RATE
        print("   🔊 voice: SAPI")
    except Exception:
        try:
            import pyttsx3
            engine = pyttsx3.init()
            print("   🔊 voice: pyttsx3")
        except Exception:
            print("   🔇 voice unavailable — "
                  "pip install pywin32")
            return

    while True:
        text = voice_queue.get()
        if text is None:
            break
        try:
            if speaker:
                speaker.Speak(text)
            elif engine:
                engine.say(text)
                engine.runAndWait()
        except Exception:
            pass


def speak(text):
    if VOICE_ON:
        voice_queue.put(text)


class VolumeLeaderboard:
    """
    Two tabs:
      Board → top 10 right now, re-ranked every 2 min
      Log   → everything that hit the board today,
              peak score and HOD preserved

    HOD marks a ticker printing its biggest
    2-MIN bar of the session
    """

    BG      = '#1e1e1e'
    BG_HEAD = '#2a2a2a'
    BG_FOOT = '#232323'
    BG_NEW  = '#1c2622'
    BORDER  = '#3a3a3a'
    TXT     = '#f0f0f0'
    TXT_DIM = '#8a8a8a'
    TXT_MID = '#b8b8b8'
    TEAL    = '#5DCAA5'
    RED     = '#F09595'
    AMBER   = '#EF9F27'

    def __init__(self):
        self.root = tk.Tk()
        self.root.title("Volume Discovery")
        self.root.geometry("500x420+40+520")
        self.root.configure(bg=self.BG)
        self.root.attributes('-topmost', True)
        self.root.resizable(True, True)

        self.running   = True
        self.next_scan = 120

        self._style()
        self._build()
        self.root.protocol("WM_DELETE_WINDOW", self.stop)

    def _style(self):
        st = ttk.Style()
        try:
            st.theme_use('clam')
        except Exception:
            pass
        st.configure('TNotebook', background=self.BG,
                     borderwidth=0)
        st.configure('TNotebook.Tab',
                     background=self.BG_FOOT,
                     foreground=self.TXT_DIM,
                     padding=[14, 5], borderwidth=0)
        st.map('TNotebook.Tab',
               background=[('selected', self.BG_HEAD)],
               foreground=[('selected', self.TXT)])
        st.configure('Treeview',
                     background=self.BG,
                     fieldbackground=self.BG,
                     foreground=self.TXT_MID,
                     rowheight=22, borderwidth=0)
        st.configure('Treeview.Heading',
                     background=self.BG_FOOT,
                     foreground='#6a6a6a',
                     borderwidth=0)
        st.map('Treeview',
               background=[('selected', self.BG_NEW)])

    def _build(self):
        head = tk.Frame(self.root, bg=self.BG_HEAD)
        head.pack(fill='x')
        tk.Label(head, text="Volume Discovery",
                 bg=self.BG_HEAD, fg=self.TXT,
                 font=('Segoe UI', 9, 'bold')
                 ).pack(side='left', padx=10, pady=6)
        self.clock = tk.Label(
            head, text="--:--:--", bg=self.BG_HEAD,
            fg=self.TXT_DIM, font=('Segoe UI', 8))
        self.clock.pack(side='right', padx=10)

        self.nb = ttk.Notebook(self.root)
        self.nb.pack(fill='both', expand=True)

        self.tab_board = tk.Frame(self.nb, bg=self.BG)
        self.tab_log   = tk.Frame(self.nb, bg=self.BG)
        self.nb.add(self.tab_board, text='Board')
        self.nb.add(self.tab_log,   text='Log')

        self._build_board()
        self._build_log()

        foot = tk.Frame(self.root, bg=self.BG_FOOT)
        foot.pack(fill='x')
        self.status_lbl = tk.Label(
            foot, text="", bg=self.BG_FOOT,
            fg='#6a6a6a', font=('Segoe UI', 8))
        self.status_lbl.pack(side='left', padx=10, pady=4)
        self.countdown = tk.Label(
            foot, text="", bg=self.BG_FOOT,
            fg='#6a6a6a', font=('Segoe UI', 8))
        self.countdown.pack(side='right', padx=10)

    def _build_board(self):
        hdr = tk.Frame(self.tab_board, bg=self.BG)
        hdr.pack(fill='x', padx=8, pady=(6, 2))
        tk.Frame(hdr, bg=self.BG, width=3).pack(side='left')
        for text, w, anchor in [
            ("SCORE", 6, 'w'), ("ACC", 5, 'w'),
            ("TICKER", 7, 'w'), ("", 4, 'w'),
            ("PRICE", 9, 'e'), ("CHG", 7, 'e'),
            ("VOLUME", 9, 'e')
        ]:
            tk.Label(hdr, text=text, bg=self.BG,
                     fg='#6a6a6a', width=w, anchor=anchor,
                     font=('Segoe UI', 8)
                     ).pack(side='left', padx=(0, 2))

        self.rows = tk.Frame(self.tab_board, bg=self.BG)
        self.rows.pack(fill='both', expand=True)
        tk.Label(self.rows, text="Waiting for 9:46AM…",
                 bg=self.BG, fg=self.TXT_DIM,
                 font=('Segoe UI', 9)).pack(pady=40)

    def _build_log(self):
        cols = ('time', 'ticker', 'peak',
                'acc', 'hod', 'price', 'chg')
        self.tree = ttk.Treeview(
            self.tab_log, columns=cols,
            show='headings', selectmode='browse')

        for c, txt, w, anc in [
            ('time',   'FIRST',  55, 'w'),
            ('ticker', 'TICKER', 65, 'w'),
            ('peak',   'PEAK',   55, 'e'),
            ('acc',    'ACC',    50, 'e'),
            ('hod',    'HOD',    40, 'center'),
            ('price',  'PRICE',  70, 'e'),
            ('chg',    'CHG',    60, 'e'),
        ]:
            self.tree.heading(c, text=txt)
            self.tree.column(c, width=w, anchor=anc)

        vs = ttk.Scrollbar(self.tab_log,
                           orient='vertical',
                           command=self.tree.yview)
        self.tree.configure(yscrollcommand=vs.set)
        self.tree.pack(side='left', fill='both',
                       expand=True, padx=(6, 0), pady=6)
        vs.pack(side='right', fill='y', pady=6)

        self.tree.tag_configure('big',
                                foreground=self.TEAL)
        self.tree.tag_configure('hod',
                                foreground=self.AMBER)

        self.tree.bind('<<TreeviewSelect>>',
                       self._log_click)

    def _log_click(self, event):
        sel = self.tree.selection()
        if sel:
            t = self.tree.item(sel[0])['values'][1]
            self._copy(str(t))

    def _copy(self, ticker):
        self.root.clipboard_clear()
        self.root.clipboard_append(ticker)
        self.status_lbl.config(text=f"copied {ticker}")

    def update_board(self, entries, new_tickers):
        for w in self.rows.winfo_children():
            w.destroy()

        if not entries:
            tk.Label(self.rows,
                     text="nothing unusual right now",
                     bg=self.BG, fg=self.TXT_DIM,
                     font=('Segoe UI', 9)).pack(pady=40)
            return

        for e in entries:
            is_new = e['ticker'] in new_tickers
            bg     = self.BG_NEW if is_new else self.BG

            row = tk.Frame(self.rows, bg=bg,
                           cursor='hand2')
            row.pack(fill='x')

            accent = self.TEAL if is_new else self.BG
            tk.Frame(row, bg=accent, width=3
                     ).pack(side='left', fill='y')

            body = tk.Frame(row, bg=bg)
            body.pack(fill='x', expand=True,
                      padx=6, pady=4)

            tk.Label(body, text=f"{e['score']:.0f}x", bg=bg,
                     fg=self.TEAL if is_new else self.AMBER,
                     width=6, anchor='w',
                     font=('Segoe UI', 10, 'bold')
                     ).pack(side='left', padx=(0, 2))

            acc = e.get('accel', 0)
            tk.Label(body, text=f"{acc:.0f}x", bg=bg,
                     fg=self.TEAL if acc >= 5
                        else self.TXT_DIM,
                     width=5, anchor='w',
                     font=('Segoe UI', 9)
                     ).pack(side='left', padx=(0, 2))

            tk.Label(body, text=e['ticker'], bg=bg,
                     fg=self.TXT, width=7, anchor='w',
                     font=('Segoe UI', 10, 'bold')
                     ).pack(side='left', padx=(0, 2))

            tk.Label(body,
                     text="HOD" if e.get('hod') else "",
                     bg=bg, fg=self.AMBER, width=4,
                     anchor='w',
                     font=('Segoe UI', 8, 'bold')
                     ).pack(side='left', padx=(0, 2))

            tk.Label(body, text=f"${e['close']:.2f}",
                     bg=bg, fg=self.TXT_MID, width=9,
                     anchor='e', font=('Segoe UI', 9)
                     ).pack(side='left', padx=(0, 2))

            pct  = e['pct_change']
            pcol = self.TEAL if pct >= 0 else self.RED
            tk.Label(body, text=f"{pct:+.1f}%", bg=bg,
                     fg=pcol, width=7, anchor='e',
                     font=('Segoe UI', 9)
                     ).pack(side='left', padx=(0, 2))

            vol_k = e['volume'] / 1000
            vtxt  = (f"{vol_k/1000:.1f}M" if vol_k >= 1000
                     else f"{vol_k:.0f}K")
            tk.Label(body, text=vtxt, bg=bg,
                     fg=self.TXT_DIM, width=9, anchor='e',
                     font=('Segoe UI', 9)
                     ).pack(side='left')

            for w in [row, body] + list(
                    body.winfo_children()):
                w.bind('<Button-1>',
                       lambda ev, t=e['ticker']:
                       self._copy(t))

    def update_log(self, log_dict):
        """Rebuild the log tab — newest first"""
        for i in self.tree.get_children():
            self.tree.delete(i)

        rows = sorted(log_dict.values(),
                      key=lambda d: d['first_seen'],
                      reverse=True)

        for d in rows:
            tags = ()
            if d['hod_hit']:
                tags = ('hod',)
            elif d['peak_score'] >= VOICE_THRESHOLD:
                tags = ('big',)

            self.tree.insert(
                '', 'end', tags=tags,
                values=(
                    d['first_seen'],
                    d['ticker'],
                    f"{d['peak_score']:.0f}x",
                    f"{d['peak_accel']:.0f}x",
                    "•" if d['hod_hit'] else "",
                    f"${d['close']:.2f}",
                    f"{d['pct_change']:+.1f}%",
                ))

        self.nb.tab(1, text=f"Log ({len(rows)})")

    def set_status(self, text):
        self.status_lbl.config(text=text)

    def tick(self):
        if not self.running:
            return
        self.clock.config(
            text=datetime.now().strftime('%H:%M:%S'))
        self.next_scan -= 1
        if self.next_scan < 0:
            self.next_scan = 120
        m, s = divmod(max(self.next_scan, 0), 60)
        self.countdown.config(text=f"next {m}:{s:02d}")
        self.root.after(1000, self.tick)

    def stop(self):
        """
        Flush peaks before closing — otherwise
        closing mid-session leaves the CSV holding
        first-appearance values
        """
        self.running = False
        try:
            flush_peaks()
        except Exception:
            pass
        voice_queue.put(None)
        try:
            self.root.destroy()
        except Exception:
            pass

    def run(self):
        self.tick()
        self.root.mainloop()


print("✅ Dashboard loaded — Board + Log tabs")
print(f"   HOD marks the biggest bar of the session")
print(f"   Voice at {VOICE_THRESHOLD}x, ticker only")

✅ Dashboard loaded — Board + Log tabs
   HOD marks the biggest bar of the session
   Voice at 15.0x, ticker only


In [ ]:
# ─── Cell 9: Scanner Loop + START ────────────────────────────

def scanner_thread(dash):
    """
    Scan loop aligned to 2-MIN bar closes
    Updates board, log tab, CSV, and voice
    """
    global board, prev_board

    reset_day()
    last_day = datetime.now().date()

    while dash.running:
        now = datetime.now()
        t   = now.time()

        if now.date() != last_day:
            reset_day()
            last_day = now.date()
            dash.root.after(
                0, dash.set_status,
                "new day — restart to rebuild baselines")

        if t >= SESSION_END:
            print("\n🔔 Session ended")
            rewrite_peaks()
            dash.root.after(0, dash.set_status,
                            "session closed")
            break

        if t < TRACK_START:
            dash.root.after(
                0, dash.set_status,
                f"waiting for {TRACK_START} · "
                f"{len(baselines)} ready")
            time_module.sleep(15)
            continue

        # ── Score ─────────────────────────────────────
        t0      = time_module.time()
        current = score_universe()
        scan_s  = time_module.time() - t0

        current_tickers = {e['ticker'] for e in current}
        new_tickers     = current_tickers - prev_board

        update_day_log(current)
        log_new_entries(current)

        board      = current
        prev_board = current_tickers

        dash.root.after(0, dash.update_board,
                        current, new_tickers)
        dash.root.after(0, dash.update_log,
                        dict(day_log))
        dash.root.after(
            0, dash.set_status,
            f"{len(baselines)} scanned · "
            f"{len(current)} on board · {scan_s:.0f}s")

        # ── Console + voice, first appearance only ────
        for e in current:
            tkr = e['ticker']

            if tkr not in printed_today:
                printed_today.add(tkr)
                hod = " HOD" if e['hod'] else "    "
                print(f"🔔 {tkr:<6} "
                      f"{e['score']:>5.0f}x hist  "
                      f"{e['accel']:>4.0f}x acc{hod}  "
                      f"${e['close']:>8.2f}  "
                      f"{e['pct_change']:>+6.1f}%  "
                      f"{e['volume']:>10,} sh  "
                      f"[{e['bar_time']}]")

            # Voice — 15x+, once per ticker per day
            if (e['score'] >= VOICE_THRESHOLD and
                    tkr not in spoken_today):
                spoken_today.add(tkr)
                speak(tkr)

        wait = seconds_to_next_bar()
        dash.next_scan = int(wait)
        time_module.sleep(wait)

    print("🏁 Scanner stopped")


def start_discovery_scanner(skip_baselines=False):
    """
    Build baselines, then launch dashboard + scanner
    skip_baselines=True reuses what's in memory
    """
    print(f"{'='*58}")
    print(f"  🔎 VOLUME DISCOVERY SCANNER")
    print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print(f"{'='*58}")

    if not skip_baselines or not baselines:
        build_baselines(universe)

    if not baselines:
        print("\n❌ No baselines — check ticker files")
        return

    if VOICE_ON:
        vt = threading.Thread(target=voice_worker,
                              daemon=True)
        vt.start()
        time_module.sleep(0.3)

    print(f"\n  Scanning {len(baselines)} tickers")
    print(f"  Board: top {BOARD_ROWS} at "
          f"{MIN_SCORE}x+ ({MIN_SCORE_HOD}x if HOD)")
    print(f"  Voice: {VOICE_THRESHOLD}x+")
    print(f"  Close window to stop")
    print(f"{'='*58}\n")

    dash = VolumeLeaderboard()

    th = threading.Thread(
        target=scanner_thread, args=(dash,),
        daemon=True)
    th.start()

    dash.run()


start_discovery_scanner()

  🔎 VOLUME DISCOVERY SCANNER
  2026-09-01 13:20

Building baselines — 124 tickers, 1 batches
   Window: 09:46:00 - 16:00:00, last 5 sessions
   Mode:   per-slot medians (time-of-day bucketed)
   Batch 1/1 (124 tickers)... 242,289 bars

   ✅ Baselines built in 27s
      Usable tickers:  124
      Skipped (thin):  0
      Skipped (<$4): 0
      Slots w/ <3 obs: 10 of 23,188 (0%) → shape fallback

      Volume shape (× all-day median):
        09:46  1.98x
        12:30  0.82x
        15:56  7.06x
   🔊 voice: SAPI

  Scanning 124 tickers
  Board: top 10 at 3.0x+ (2.0x if HOD)
  Voice: 15.0x+
  Close window to stop



C:\Users\kwgit\AppData\Local\Temp\ipykernel_16636\3234249102.py:147: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g['volume']
C:\Users\kwgit\AppData\Local\Temp\ipykernel_16636\3234249102.py:152: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  prior_bars = df.groupby('symbol').apply(


🔔 TRI        9x hist     7x acc      $  106.15    -1.1%      22,829 sh  [13:06]
🔔 FTNT       4x hist     2x acc      $  162.11    -5.2%      28,642 sh  [13:06]
🔔 NVDL       3x hist     3x acc      $   34.04    -2.1%      81,412 sh  [13:06]
🔔 FANG       3x hist     2x acc      $  202.25    +0.9%      10,852 sh  [13:06]


C:\Users\kwgit\AppData\Local\Temp\ipykernel_16636\3234249102.py:147: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g['volume']
C:\Users\kwgit\AppData\Local\Temp\ipykernel_16636\3234249102.py:152: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  prior_bars = df.groupby('symbol').apply(
C:\Users\kwgit\AppData\Local\Temp\ipykernel_16636\3234249102.py:147: DeprecationWarning: DataFrameGroupBy.appl

🔔 AXON      15x hist     2x acc      $  515.18    -9.1%      10,134 sh  [13:08]
🔔 AAPU       6x hist     3x acc      $   42.75    +5.0%      11,636 sh  [13:08]
🔔 AAPL       6x hist     2x acc      $  325.05    +2.5%     378,888 sh  [13:08]
🔔 CSX        4x hist     1x acc      $   48.12    -4.7%      39,378 sh  [13:08]
🔔 ISRG       4x hist     3x acc      $  367.42    -2.5%      11,954 sh  [13:08]
🔔 ADBE       4x hist     2x acc      $  288.20    -1.6%      23,238 sh  [13:08]
🔔 SBUX       3x hist     2x acc      $  105.60    -0.6%      28,709 sh  [13:08]
🔔 CEG        3x hist     2x acc      $  276.67    +0.7%      10,037 sh  [13:08]


...

🔔 MCHP      56x hist    59x acc      $   71.40    -2.8%     558,698 sh  [13:14]
🔔 MELI      19x hist    18x acc HOD  $ 1945.05    +0.5%      18,288 sh  [13:14]
🔔 PANW       4x hist     1x acc      $  359.46    -5.9%      27,831 sh  [13:14]


C:\Users\kwgit\AppData\Local\Temp\ipykernel_16636\3234249102.py:147: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g['volume']
C:\Users\kwgit\AppData\Local\Temp\ipykernel_16636\3234249102.py:152: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  prior_bars = df.groupby('symbol').apply(


🔔 GME      334x hist   388x acc HOD  $   18.46    +0.4%   3,025,567 sh  [13:16]
🔔 WMT        6x hist     8x acc      $  106.16    +1.3%     306,779 sh  [13:16]


C:\Users\kwgit\AppData\Local\Temp\ipykernel_16636\3234249102.py:147: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g['volume']
C:\Users\kwgit\AppData\Local\Temp\ipykernel_16636\3234249102.py:152: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  prior_bars = df.groupby('symbol').apply(


🔔 LITE       4x hist     3x acc      $  863.81    -5.6%      36,845 sh  [13:18]
🔔 SNDQ       4x hist     2x acc      $   15.16    +0.3%     139,691 sh  [13:18]
🔔 ORLY       3x hist     4x acc      $   87.83    -1.1%      25,094 sh  [13:18]


In [ ]:
# ─── Cell 10: End of Day Review ──────────────────────────────

def review_today():
    """What hit the board today, in order of appearance"""
    if not os.path.exists(LOG_FILE):
        print("❌ No log yet")
        return

    df    = pd.read_csv(LOG_FILE)
    today = datetime.now().strftime('%Y-%m-%d')
    df    = df[df['timestamp'].str.startswith(today)]

    print(f"\n{'='*58}")
    print(f"  VOLUME DISCOVERY — {today}")
    print(f"  {len(df)} tickers hit the board")
    print(f"{'='*58}\n")

    if len(df) == 0:
        print("  Nothing today")
        return

    print(df[['bar_time', 'ticker', 'score',
              'close', 'pct_change', 'volume',
              'direction']].to_string(index=False))

    print(f"\n  Highest scores:")
    top = df.nlargest(5, 'score')
    for _, r in top.iterrows():
        print(f"  {r['ticker']:<6} "
              f"{r['score']:>6.0f}x at {r['bar_time']}  "
              f"${r['close']:.2f}  "
              f"{r['pct_change']:+.1f}%")

    print(f"\n  Fill in notes column — was any of it "
          f"tradeable?")
    print(f"  {LOG_FILE}")


review_today()


  VOLUME DISCOVERY — 2026-08-31
  14 tickers hit the board

bar_time ticker  score  close  pct_change  volume direction
   11:30   AAPU    5.2  39.95        -3.5   16453      down
   11:30   DKNG    3.6  24.50        -3.0   35990        up
   11:30   SNDQ    3.6  16.68        -2.0  220277        up
   11:30    GME    3.2  18.60         4.1   29355      down
   11:34   SKHX    4.3  13.70         1.8   15036      down
   11:48   AMZU   13.9  37.93        -4.2   17146        up
   11:52   MVLL    4.9  23.87        -4.9   17087      down
   11:58   MRVL    9.4 210.19        -3.0  467423        up
   12:00   IBIT    4.9  44.58         1.5  712857        up
   12:08   TSLL    5.7   9.88        10.0  801016        up
   12:12   SNAP    3.2   5.55         2.0  273090        up
   12:14   SPCH    6.4   9.38         3.0  313653      down
   12:24   SSPC    4.2  10.24        -2.9   96476      down
   12:26   SNXX    3.5  13.06         1.6  427936        up

  Highest scores:
  AMZU       14x at 